# 🛡️ Kaggle 24/7 Full Web Platform (Fast Startup & Cloudflare Tunnel)
Notebook ultra-rapide et résilient avec Cloudflare Tunnel HTTPS, auto-sauvegarde SQLite et gestionnaire de mémoire 24/7.

In [ ]:
# 1. Nettoyage et installation ultra-rapide des dépendances Python & Cloudflare Tunnel
!rm -rf /kaggle/working/projet_osint
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!git clone https://github.com/your-repo/projet_osint.git /kaggle/working/projet_osint
%cd /kaggle/working/projet_osint/backend
!pip install --no-cache-dir -r requirements.txt
!playwright install chromium --with-deps || true

In [ ]:
# 2. Démarrage du Serveur FastAPI (servant l'Interface Web et l'API) + Tunnel Cloudflare HTTPS
import subprocess
import time
import re

print('Démarrage du Serveur FastAPI (Frontend + Backend) sur port 8000...')
server_process = subprocess.Popen(['python', '-m', 'app.main'])
time.sleep(6)

print('Lancement du Tunnel Cloudflare HTTPS...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for _ in range(20):
    line = tunnel_process.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n======================================================')
            print(f'🚀 VOTRE INTERFACE WEB EST EN LIGNE H24 : {match.group(0)}')
            print('======================================================\n')
            url_found = True
            break
    time.sleep(1)

if not url_found:
    print('Tunnel Cloudflare initialisé (Consultez les logs cloudflared pour l URL).')

In [ ]:
# 3. Boucle d'exécution continue 24/7 avec Checkpoints SQLite horaires et Garbage Collector
import time
from app.cloud_sync.kaggle_persistence import KagglePersistenceManager
from app.cloud_sync.garbage_collector import GarbageCollectorManager

print('🟢 Boucle d\'exécution continue 24/7 active...')
for hour in range(1, 11):
    time.sleep(3600)
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] Checkpoint State Heure {hour}/10...')
    GarbageCollectorManager.cleanup_temp_storage()
    KagglePersistenceManager.checkpoint_state()